# Ion Bernstein Waves

**Ion Bernstein Waves (IBW)** are electrostatic eigenmodes of a magnetised warm plasma propagating perpendicular to $\mathbf{B}_0$. They exist near harmonics of the ion cyclotron frequency $\Omega_i$ and are of practical interest for RF heating in fusion devices.

A 1D2v Vlasov–Poisson simulation is run in a uniform background field, and the density fluctuation spectrum $S(k,\omega)$ is extracted to identify the IBW branches and compare them against the cold-plasma and warm-plasma dispersion relations.

In [ ]:
include("../scripts/select_backend.jl")
bslLD.greet()

## Simulation Setup

A 1D2v phase space is used: $x\in[0,10]$ with $N_x=64$ grid points, $v_\perp\in[-6,6]^2$ with $N_v=32\times32$ velocity grid points. The background magnetic field sets the ion gyro-frequency $\Omega_i=1$. A Maxwellian distribution is perturbed by a tiny random noise ($\epsilon=10^{-7}$) to seed all Fourier modes. The time step is $\Delta t=0.025$ and the simulation runs to $T=300$ (12\,000 steps), long enough to resolve IBW branches up to the 5th harmonic.

In [ ]:
grid =  bslLD.Grid([0.0,-6.0,-6.0],[10.0,6.0,6.0],[64,32,32],1, 1.0, 3)
simTime = bslLD.SimulationTime(0.025, 300.0, gyro_frequency=1.0)


# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
initFuncx(x) = 1+ 0.0000001 * rand()
f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);


In [ ]:
print(Array(grid.vaxes[1]))

In [ ]:
mutable struct Diag
    rho::Vector
end
Diag() = Diag([])

function diags!(diags, f, rho, Ex, grid, simTime)
    simTime.step % 1 == 0 || return
    push!(diags.rho, copy(rho.data[:]))
end

function stepStrang!(f, grid, simTime, diag)
    phase_start = simTime.phase
    Ω = simTime.gyro_frequency

    # V half-step at phase(t)
    sol = bslLD.solve_fields(bslLD.Moments(bslLD.compute_density(f, grid)), grid, bslLD.PoissonSolver(-0.001))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f, grid, simTime, sol.E)

    # X full-step at phase(t + dt/2)
    simTime.phase = phase_start + Ω * simTime.dt * 0.5
    simTime.fraction_dt = 1.0
    bslLD.advectX!(f, grid, simTime)

    rho = bslLD.compute_density(f, grid)

    # V half-step at phase(t + dt)
    simTime.phase = phase_start + Ω * simTime.dt
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.PoissonSolver(-0.001))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0

    # restore so advance!() applies the correct full-step increment
    simTime.phase = phase_start

    diags!(diag, f, rho, sol.E[1], grid, simTime)
end;


In [ ]:
bslLD.ProgressMeter.ijulia_behavior(:clear);

In [ ]:
diags = Diag()
while bslLD.continue_advection(simTime, true)
    stepStrang!(f, grid, simTime, diags)
    bslLD.advance!(simTime)
end

## Spectral Analysis

The density fluctuation time series $\delta n(x,t)$ is Fourier-transformed in both space and time to obtain the $(k,\omega)$ power spectrum $S(k,\omega) = |\hat{n}(k,\omega)|^2$.

In [ ]:
using FFTW, DSP, Statistics, CairoMakie

## $\omega$–$k$ Spectrum

The power spectrum $S(k,\omega) = |\hat{n}(k,\omega)|^2$ reveals the IBW dispersion branches as bright ridges. Peaks should align with harmonics of the ion cyclotron frequency $n\Omega_i$ (horizontal lines at $\omega = n$). The overlaid red curve shows the warm-plasma lower-hybrid/IBW approximation $\omega = \sqrt{\omega_{LH}^2 + \Omega_i^2 + 3k^2 v_{th}^2}$ (with $\omega_{LH}^2=1000$ for these parameters, $v_{th}=1$, $\Omega_i=1$). A Kaiser window is applied in space before the FFT to reduce spectral leakage.

In [ ]:
locData = transpose(hcat(map(x-> x.-mean(x), Array.(diags.rho))...))

Nx, Ny = size(locData)


omega = fftfreq(size(locData, 1), 1) / simTime.dt*2*pi
k = (fftfreq(size(locData, 2))*length(grid.xaxes[1]) *2*pi/grid.max[1])[1:round(Int,Ny/2)]


w = kaiser(Ny, 3)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

nOmegaMax = round(Int,size(locData, 1) ÷ 4)

fig, ax, plt = heatmap(
    k,
    omega[1:nOmegaMax],
    transpose(log.(abs.(fft(windowed))[1:nOmegaMax, 1:round(Int, Ny/2)])),
        axis = (
        xlabel = "k",
        ylabel = "ω",
        title = "Fourier Interpolation")
)

lines!(
    ax,
    k,
    sqrt.(1000 .+ 1 .+ 3 .* k.^2),
    label = "√(ω_LH² + 1 + 3k²)",
    color = :red
)

limits!(
    ax,
    nothing, nothing,
    0, maximum(omega[1:nOmegaMax])
)

axislegend(ax, position = :rt)



fig